# Colab 24e — consistent current-string labels + provenance audit + restored embedding-space suite

**Relationship to 24d.** colab24d already established *one metric everywhere* (P1): high-pair
selection, AUROC labels, the model input, and the full-pool oracle all judge similarity with the
**same** `normLev` recomputed on the **current** pool strings (AA, SS, *and* 3Di). colab24e keeps
that protocol **unchanged** — same synthetic AA/SS training recipe, architecture, seeds, epochs; the
same per-feed AA/SS/3Di pools; the same `encode_pad` overflow guard; the same pair-source loading
(`cath_s20_pairs_sample` + `cath_s20_pairs_high`) + unordered-pair dedup; the same full-pool oracle;
the same retrieval metrics, bootstrap CIs, and normalized length baseline. Legacy
`cath_eval.csv.gz` is left untouched.

**What 24e adds — only these three things:**

1. **Richer score provenance.** Every feed eval table now carries **both** `source_aa_score` and
   `source_ss_score` (preserved supervisor data, used for provenance checks only), alongside
   `current_normlev` (the sole ground-truth score) and `is_high = current_normlev >= 0.70`.
2. **An explicit supplied-vs-recomputed audit cell** — for AA and SS: median / p95 / maximum absolute
   `|source - current|` difference, plus the count of high↔non-high **flips** at the 0.70 cut.
3. **The full colab24c embedding-space visual suite**, recomputed from 24e's feed-matched data: the
   AUROC bar chart, the MAP@10-vs-length-baseline chart, linear probes, the PCA spectrum + axis
   correlations, the PCA-truncation rate-distortion curve, the AA-encoder UMAP (high-AA pairs as red
   segments — **overlay drawn from the feed-matched AA eval table, not the legacy CSV**), neighbour
   tables, the displacement probe, and the local zoom.

**The one consistent-label sentence.** *For each representation, both pairwise discrimination (AUROC)
and full-pool retrieval are evaluated against the same normalized-Levenshtein definition computed
from the current feed-specific strings.* colab24e adds transparent provenance auditing and the
visual analysis needed to explain the resulting embedding geometry.

**Do not read 24d→24e differences as model improvement or decline.** The model, pools, query logic,
oracle, and metrics are identical; 24e only adds provenance columns, an audit, and figures. (24d
remains valuable as the feed-matched run; 24e is its consistent-label, fully-audited successor.)

| feed | score used (sole ground truth) | high cut | retrieval queries |
|---|---|---|---|
| AA  | recomputed `normLev` on current strings | `>= 0.70` | endpoints of high-AA pairs |
| SS  | recomputed `normLev` on current strings | `>= 0.70` | endpoints of high-SS pairs |
| 3Di | recomputed `normLev` on current strings | `>= 0.70` | endpoints of high-3Di pairs |


## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
# Source pair files (the canonical labels) + pools + the legacy eval (kept for comparison only).
for f in ['cath_s20_pairs_sample.csv.gz', 'cath_s20_pairs_high.csv.gz',
          'cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz',
          'cath_s20_3di.csv.gz', 'cath_eval.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch torchvision rapidfuzz scikit-learn scipy umap-learn matplotlib --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLevenshtein
from rapidfuzz.process import cdist as rf_cdist

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Constants (encoder recipe identical to colab24c / colab19 / colab17b)

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
SS_SET = set(SS_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN = 50
MAX_LEN_SEQ = 200
MAX_LEN = 200

N_TRAIN = 30000
NUM_EPOCHS = 30
BATCH_SIZE = 128
K = 16

BAND_LOW_AA = 0.30
BAND_LOW_SS = 0.56
BAND_HIGH   = 0.70          # high-sim cutoff: positives for AUROC + oracle true-set bar + query selection
BIN_NAMES = ['far', 'mid', 'high']

K_RETRIEVAL = 10
SEED_TRAIN_AA = 42
SEED_TRAIN_SS = 43

print(f'AA encoder BAND_LOW={BAND_LOW_AA} | SS encoder BAND_LOW={BAND_LOW_SS} '
      f'| high-sim cutoff={BAND_HIGH} | retrieval @k={K_RETRIEVAL} | seed={SEED}')

## 3. Helpers — Levenshtein, encode, perturb (RNG threaded for determinism)

In [ ]:
def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLevenshtein.distance(a, b) / L

def is_standard_aa(seq): return all(c in AA_SET for c in seq)
def is_standard_ss(seq): return all(c in SS_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    if len(seq) > max_len:
        raise ValueError(f'sequence length {len(seq)} exceeds max_len={max_len} '
                         f'(pool is filtered to <= {max_len}; unfiltered input?)')
    idx = [CHAR_TO_IDX[c] for c in seq]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, alphabet, rng, max_len=MAX_LEN):
    s = list(seq); abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); choices = [c for c in abc if c != s[i]]; s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        elif op == 'del':
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def random_seq(alphabet, rng, min_len=MIN_LEN, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(alphabet), size=length))

def bin_idx_for(x, band_low):
    if x < band_low: return 0
    if x < BAND_HIGH: return 1
    return 2

def make_full_range_pairs(alphabet, n_pairs, rng):
    pairs = []; attempts = 0; max_attempts = n_pairs * 4
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = random_seq(alphabet, rng)
        if len(seed) < MIN_LEN: continue
        L = len(seed); t = float(rng.uniform(0.0, 1.0)); k = max(0, int(round((1.0 - t) * L)))
        other = perturb(seed, k, alphabet, rng)
        if 1 <= len(other) <= MAX_LEN:
            pairs.append((seed, other, norm_lev(seed, other)))
    return pairs

## 4. Architecture + training (3-bin CE head, identical to colab24c)

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, K, vocab_size=VOCAB_SIZE, embed_dim=32,
                 hidden1=32, hidden2=64, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.K = K; self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.pool  = nn.AdaptiveAvgPool1d(K)
        self.fc    = nn.Linear(hidden2 * K, out_dim)

    def forward(self, x):
        mask = (x != self.pad_idx).float()
        e = self.embedding(x).permute(0, 2, 1)
        h = F.relu(self.conv1(e)); h = F.relu(self.conv2(h))
        h = h * mask.unsqueeze(1)
        h = self.pool(h).flatten(1)
        return F.normalize(self.fc(h), p=2, dim=1)


class SiameseClassifier(nn.Module):
    def __init__(self, K, embed_out=128, hidden_mlp=64, n_bins=3):
        super().__init__()
        self.encoder = SiameseEncoder(K)
        self.head = nn.Sequential(
            nn.Linear(embed_out, hidden_mlp), nn.LeakyReLU(0.01),
            nn.Linear(hidden_mlp, n_bins))
    def forward(self, a, b):
        return self.head(torch.abs(self.encoder(a) - self.encoder(b)))
    def encode(self, x):
        return self.encoder(x)


class PairDatasetCE(Dataset):
    def __init__(self, pairs, band_low):
        self.a = [encode_pad(a) for a, _, _ in pairs]
        self.b = [encode_pad(b) for _, b, _ in pairs]
        self.bins = torch.tensor([bin_idx_for(l, band_low) for _, _, l in pairs], dtype=torch.long)
    def __len__(self): return len(self.bins)
    def __getitem__(self, i): return self.a[i], self.b[i], self.bins[i]


def train_encoder(alphabet, band_low, label, train_seed, n_epochs=NUM_EPOCHS):
    torch.manual_seed(train_seed)
    rng = np.random.default_rng(train_seed)
    print(f'\n===== Training {label} encoder (alphabet={alphabet}, band_low={band_low}, seed={train_seed}) =====')
    pairs = make_full_range_pairs(alphabet, N_TRAIN, rng)
    labels = np.array([p[2] for p in pairs])
    counts = {b: int(c) for b, c in zip(BIN_NAMES,
              np.bincount([bin_idx_for(l, band_low) for l in labels], minlength=3))}
    print(f'  {len(pairs)} pairs, normLev in [{labels.min():.3f}, {labels.max():.3f}], bins={counts}')
    dl = DataLoader(PairDatasetCE(pairs, band_low), batch_size=BATCH_SIZE, shuffle=True)
    model = SiameseClassifier(K).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(1, n_epochs + 1):
        tot = 0.0; nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        if epoch % 5 == 0 or epoch == 1:
            print(f'  [{label}] epoch {epoch:3d}/{n_epochs} CE={tot/nb:.4f}')
    model.eval()
    return model

## 5. Per-representation pools (genuinely per-feed, identical to colab24c)

Each representation is filtered **only** by its own validity (standard alphabet + length in `[MIN_LEN, MAX_LEN]`); a domain's eligibility in one representation never depends on another. `RESCUED` keeps the sub-`MIN_LEN` high-AA pair wherever its sequence is otherwise valid.

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))
raw = pd.concat([train_df, test_df], ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_3di.csv.gz'))

RESCUED = {'4z0mC02', '3qkaE02'}

def _valid(seq, is_std, domain):
    return (isinstance(seq, str) and is_std(seq)
            and ((MIN_LEN <= len(seq) <= MAX_LEN) or domain in RESCUED))

id_to_aa  = {d: s for d, s in zip(raw['domain_id'],   raw['aa_seq'])            if _valid(s, is_standard_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'],   raw['ss_seq'])            if _valid(s, is_standard_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_standard_aa, d)}

POOL_IDS  = {'AA': list(id_to_aa), 'SS': list(id_to_ss), '3Di': list(id_to_3di)}
LOOKUP    = {'AA': id_to_aa,       'SS': id_to_ss,       '3Di': id_to_3di}
SCORE_COL = {'AA': 'aa_score',     'SS': 'ss_score',     '3Di': '3di_score'}
POOL_SET  = {f: set(POOL_IDS[f]) for f in POOL_IDS}

for f in ['AA', 'SS', '3Di']:
    print(f'  {f:<4} pool = {len(POOL_IDS[f]):>6}')

## 6. Feed-matched eval tables — one metric, full provenance

Load the two raw pair-source files, combine, **deduplicate as unordered pairs** (so `(A,B)` and
`(B,A)` cannot both occur), then build three eval tables **independently** — each filtered to *its
own* pool and scored by recomputing `normLev` on the *current* strings in *its own* alphabet. No AA
bin is allowed to select an SS or 3Di pair.

**24e schema (vs 24d).** Each feed table keeps **both** supplied source scores — `source_aa_score`
and `source_ss_score` — as provenance, plus `current_normlev` (the **sole** score used for `is_high`,
query selection, AUROC labels, and oracle relevance) and `is_high = current_normlev >= 0.70`.

> **Column note.** The pair-source files are **headerless** — the first data row was historically
> mis-read as a header. Column order verified against `cath_eval.csv.gz` (correlation 1.000, zero
> max-diff): **col2 = `ss_score`, col3 = `aa_score`**. The remaining columns (`src_*`) are the source
> files' own *local-alignment* metrics and are **not** used (cf. `memory/data_provenance_census.md`:
> the source `_LEV_filtered` alignment_score is a local metric ≠ our global normLev).


In [ ]:
# --- load + combine + dedup unordered ---
PAIR_COLS = ['domain_a', 'domain_b', 'ss_score', 'aa_score', 'src4', 'src5', 'src_flag']
samp = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_pairs_sample.csv.gz'), header=None, names=PAIR_COLS)
high = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_pairs_high.csv.gz'), header=None, names=PAIR_COLS)

src_pairs = pd.concat([samp, high], ignore_index=True)
N_SOURCE_RAW = len(src_pairs)
src_pairs = src_pairs[src_pairs['domain_a'] != src_pairs['domain_b']]   # defensive: drop self-pairs
src_pairs['pair_key'] = [frozenset((a, b)) for a, b in zip(src_pairs['domain_a'], src_pairs['domain_b'])]
src_pairs = src_pairs.drop_duplicates('pair_key').reset_index(drop=True)
N_SOURCE_DEDUP = len(src_pairs)
print(f'source pairs: raw concat = {N_SOURCE_RAW:,}  ->  unordered-deduped = {N_SOURCE_DEDUP:,}')

# sanity check the column mapping against the legacy eval (corr should be 1.000, max-diff 0.0)
_ev = pd.read_csv(os.path.join(DATA_DIR, 'cath_eval.csv.gz'))
_ev['pair_key'] = [frozenset((a, b)) for a, b in zip(_ev['domain_a'], _ev['domain_b'])]
_m = _ev.merge(src_pairs, on='pair_key', suffixes=('_ev', ''))
print(f'column-mapping check vs cath_eval (n={len(_m)}):')
print(f"  aa: corr={_m['aa_score_ev'].corr(_m['aa_score']):.4f}  max|diff|={(_m['aa_score_ev']-_m['aa_score']).abs().max():.4f}")
print(f"  ss: corr={_m['ss_score_ev'].corr(_m['ss_score']):.4f}  max|diff|={(_m['ss_score_ev']-_m['ss_score']).abs().max():.4f}")
print('  (expect corr ~1.0000 and max|diff| ~0.0000 -> col2=ss_score, col3=aa_score confirmed)')

In [ ]:
# --- build the three feed eval tables, independently ---
# CANONICAL METRIC (P1, unchanged from 24d): exact normLev recomputed on the CURRENT pool strings,
# for ALL feeds (AA, SS, 3Di). It is the SAME metric the full-pool oracle uses, so high-pair
# selection, AUROC labels, the model input, and the retrieval oracle all judge similarity identically
# on the same strings.
#
# 24e schema change (provenance only): every feed table keeps BOTH supplied source scores
# (source_aa_score, source_ss_score) for transparency, plus current_normlev (the SOLE score used for
# labels / selection / AUROC / oracle) and is_high = current_normlev >= BAND_HIGH.

def build_feed_eval(feed):
    """Eligible = both endpoints in this feed's pool. Score = recomputed normLev on current strings.
    is_high = current_normlev >= BAND_HIGH. Independent per feed; no cross-feed bin selection.
    Both supplied source scores are preserved as provenance and are NEVER used for selection."""
    inpool = POOL_SET[feed]; lk = LOOKUP[feed]
    sub = src_pairs[src_pairs['domain_a'].isin(inpool) & src_pairs['domain_b'].isin(inpool)]
    a = sub['domain_a'].values; b = sub['domain_b'].values
    current = np.array([norm_lev(lk[x], lk[y]) for x, y in zip(a, b)])
    out = pd.DataFrame({
        'domain_a': a, 'domain_b': b,
        'source_aa_score': sub['aa_score'].astype(float).values,   # provenance only
        'source_ss_score': sub['ss_score'].astype(float).values,   # provenance only
        'current_normlev': current,                                # SOLE ground-truth score
    })
    out['is_high'] = (out['current_normlev'] >= BAND_HIGH).astype(int)
    return out.reset_index(drop=True)

EVAL = {feed: build_feed_eval(feed) for feed in ['AA', 'SS', '3Di']}

# --- export successor tables (legacy cath_eval.csv.gz AND the 24d *_perrep tables left untouched) ---
OUT_NAME = {'AA': 'cath_eval_aa_perrep_24e.csv.gz',
            'SS': 'cath_eval_ss_perrep_24e.csv.gz',
            '3Di': 'cath_eval_3di_perrep_24e.csv.gz'}
for feed in ['AA', 'SS', '3Di']:
    path = os.path.join(DATA_DIR, OUT_NAME[feed])
    EVAL[feed].to_csv(path, index=False)
    print(f'wrote {OUT_NAME[feed]:<30} rows={len(EVAL[feed]):>6}  high={int(EVAL[feed]["is_high"].sum()):>5}  -> {path}')
print('\ncolumns:', list(EVAL['AA'].columns))
print('NOTE: new files (suffix _24e); do NOT overwrite cath_eval.csv.gz or the 24d *_perrep tables'
      ' (see memory/feedback_dataset_rollback.md).')


### 6b. Provenance audit — supplied source score vs current recomputed normLev

For AA and SS, quantify how far the supervisor-supplied score drifts from a fresh recompute on the
current strings: median / p95 / maximum absolute difference, and the number of pairs that **flip**
across the 0.70 high-sim cut. `current_normlev` is the sole score used everywhere; the supplied
scores are provenance only. (3Di has no supplied own-alphabet score in the pair files.)


In [ ]:
print('=' * 88)
print('PROVENANCE AUDIT — supplied source score vs current_normlev (each feed, own alphabet)')
print('=' * 88)
SELF_SUPPLIED = {'AA': 'source_aa_score', 'SS': 'source_ss_score'}   # 3Di has no own supplied score
for feed in ['AA', 'SS']:
    e = EVAL[feed]
    supplied = e[SELF_SUPPLIED[feed]].values.astype(float)
    current  = e['current_normlev'].values.astype(float)
    ok = ~np.isnan(supplied)
    diff = np.abs(supplied[ok] - current[ok])
    hi_supp = supplied[ok] >= BAND_HIGH
    hi_curr = current[ok]  >= BAND_HIGH
    flips = int((hi_supp != hi_curr).sum())
    print(f'\n{feed} feed  (n={int(ok.sum())} pairs with a supplied {feed.lower()}_score)')
    print(f'  |supplied - current|   median={np.median(diff):.4f}   p95={np.percentile(diff, 95):.4f}'
          f'   max={diff.max():.4f}')
    print(f'  high(>=0.70)           supplied={int(hi_supp.sum()):>4}   current={int(hi_curr.sum()):>4}'
          f'   | flips across 0.70 = {flips}')
    if flips:
        flip_rows = e[ok].iloc[np.where(hi_supp != hi_curr)[0]]
        for _, r in flip_rows.head(10).iterrows():
            print(f'     flip: {r["domain_a"]} <-> {r["domain_b"]}  '
                  f'supplied={r[SELF_SUPPLIED[feed]]:.4f}  current={r["current_normlev"]:.4f}')
print('\n3Di feed: no supplied own-alphabet score in the pair files -> recompute only, nothing to audit.')
print('\n=> current_normlev is the SOLE score used for labels, query selection, AUROC, and oracle relevance;')
print('   source_aa_score / source_ss_score are retained for provenance only.')


## 7. Audit table + scope statement

Per-feed: source-pair count, eligible-pair count, high-pair count, unique query count, pool size, seed. Plus an old-vs-new comparison so the SS power gain is explicit.

In [ ]:
def feed_high(feed):
    e = EVAL[feed]
    return e[e['is_high'] == 1]

def feed_queries(feed):
    h = feed_high(feed)
    return sorted(set(h['domain_a']) | set(h['domain_b']))

print('=' * 96)
print('AUDIT — feed-matched eval construction (all counts are AFTER feed-specific filtering of the')
print('         supplied pair-source SAMPLE; NOT all C(14907,2) possible CATH pairs)')
print('=' * 96)
print(f'{"feed":<6}{"source_pairs":>14}{"eligible":>11}{"high>=0.70":>12}{"queries":>9}{"pool":>9}{"seed":>6}')
print('-' * 96)
QUERIES = {}
for feed in ['AA', 'SS', '3Di']:
    q = feed_queries(feed); QUERIES[feed] = q
    print(f'{feed:<6}{N_SOURCE_DEDUP:>14,}{len(EVAL[feed]):>11,}{int(feed_high(feed).shape[0]):>12,}'
          f'{len(q):>9,}{len(POOL_IDS[feed]):>9,}{SEED:>6}')
print('-' * 96)
print('Scope: "all high pairs" = all high pairs in the supplied pair-source files after feed-specific')
print('       filtering. The pair files are a SAMPLE (~57k unordered pairs), not an exhaustive enumeration.')

# --- old vs new: what the AA-centred cath_eval surfaced per feed ---
print('\n--- legacy cath_eval.csv.gz vs feed-matched (high-pair / unique-query counts) ---')
_ev3 = pd.read_csv(os.path.join(DATA_DIR, 'cath_eval.csv.gz'))
_ev3['3di_score'] = [norm_lev(id_to_3di[a], id_to_3di[b]) if a in id_to_3di and b in id_to_3di else np.nan
                     for a, b in zip(_ev3['domain_a'], _ev3['domain_b'])]
print(f'{"feed":<6}{"legacy high":>13}{"legacy queries":>16}{"new high":>10}{"new queries":>13}')
for feed in ['AA', 'SS', '3Di']:
    sc = SCORE_COL[feed]
    le = _ev3.dropna(subset=[sc])
    le = le[le['domain_a'].isin(POOL_SET[feed]) & le['domain_b'].isin(POOL_SET[feed])]
    lh = le[le[sc] >= BAND_HIGH]
    lq = sorted(set(lh['domain_a']) | set(lh['domain_b']))
    print(f'{feed:<6}{lh.shape[0]:>13,}{len(lq):>16,}{feed_high(feed).shape[0]:>10,}{len(QUERIES[feed]):>13,}')

## 8. Train the two encoders (frozen afterwards)

In [ ]:
model_aa = train_encoder(AA_ALPHABET, BAND_LOW_AA, 'AA', SEED_TRAIN_AA)
model_ss = train_encoder(SS_ALPHABET, BAND_LOW_SS, 'SS', SEED_TRAIN_SS)

## 9. Metric machinery — predicted L2-sim, AUROC over each feed's own eligible set

In [ ]:
@torch.no_grad()
def encode_pool(model, seq_lookup, ids, batch_size=256):
    model.eval(); valid = [d for d in ids if d in seq_lookup]; out = []
    for i in range(0, len(valid), batch_size):
        x = torch.stack([encode_pad(seq_lookup[d]) for d in valid[i:i+batch_size]]).to(device)
        out.append(model.encoder(x))
    return (torch.cat(out, 0) if out else torch.empty(0)), valid

@torch.no_grad()
def pred_sim_strings(model, A, B, batch=512):
    model.eval(); sims = []
    for i in range(0, len(A), batch):
        xa = torch.stack([encode_pad(s) for s in A[i:i+batch]]).to(device)
        xb = torch.stack([encode_pad(s) for s in B[i:i+batch]]).to(device)
        d = torch.linalg.vector_norm(model.encoder(xa) - model.encoder(xb), dim=1)
        sims.append((1.0 - d / 2.0).cpu().numpy())
    return np.concatenate(sims) if sims else np.array([])

def auroc_cell(model, feed):
    """is-high AUROC over THIS feed's full eligible set, using THIS feed's own score bins."""
    sc, lk = SCORE_COL[feed], LOOKUP[feed]
    sub = EVAL[feed]
    y = sub['is_high'].values
    if y.sum() == 0 or y.sum() == len(y):
        return np.nan, int(y.sum()), len(y)
    pred = pred_sim_strings(model, [lk[d] for d in sub['domain_a']], [lk[d] for d in sub['domain_b']])
    return float(roc_auc_score(y, pred)), int(y.sum()), len(y)

## 10. Oracle — full-pool exact-Levenshtein neighbour sets (unchanged from colab24c)

The full-pool exact-Levenshtein oracle *stays as it is*: it independently recomputes the true `>= 0.70` neighbour set per query over the whole pool. (Queries now come from the feed-matched high pairs of §6.)

In [ ]:
def build_oracle_cache(feed):
    """Exact-Levenshtein true neighbour sets (>= BAND_HIGH) per query, over the full pool."""
    lk = LOOKUP[feed]
    pool_ids = POOL_IDS[feed]
    idx = {d: i for i, d in enumerate(pool_ids)}
    pool_seqs = [lk[d] for d in pool_ids]
    lens = np.array([len(s) for s in pool_seqs])
    q_ids = QUERIES[feed]
    if not q_ids:
        return {'pool_ids': pool_ids, 'idx': idx, 'q_ids': [], 'true_sets': {}}
    D = rf_cdist([lk[q] for q in q_ids], pool_seqs,
                 scorer=RFLevenshtein.distance, workers=-1).astype(float)
    true_sets = {}
    for r, q in enumerate(q_ids):
        qi = idx[q]
        denom = np.maximum(lens, lens[qi]); denom[denom == 0] = 1
        osim = 1.0 - D[r] / denom; osim[qi] = -np.inf
        true_sets[q] = set(np.where(osim >= BAND_HIGH)[0].tolist())
    nt = [len(true_sets[q]) for q in q_ids]
    print(f'  oracle[{feed:<3}]: {len(q_ids):>3} queries, median |T|={int(np.median(nt))}, max={max(nt)}')
    return {'pool_ids': pool_ids, 'idx': idx, 'q_ids': q_ids, 'true_sets': true_sets}

print('Building exact-Levenshtein oracle neighbour sets (feed-matched queries)...')
ORACLE = {feed: build_oracle_cache(feed) for feed in ['AA', 'SS', '3Di']}

# Invariant (P1): selection and the oracle now use the SAME normLev metric, so every feed-matched query
# (an endpoint of a >=0.70 pair whose partner is in the pool) MUST have at least one oracle true neighbour.
for feed in ['AA', 'SS', '3Di']:
    ts = ORACLE[feed]['true_sets']
    empty = [q for q in ORACLE[feed]['q_ids'] if len(ts[q]) == 0]
    assert not empty, f'{feed}: {len(empty)} query/queries have an EMPTY oracle true set (metric mismatch): {empty[:5]}'
    print(f'  [{feed:<3}] all {len(ORACLE[feed]["q_ids"]):>3} queries have >= 1 oracle neighbour  OK')

# Stronger 24e invariant: each high pair's designated PARTNER must itself appear in the query's
# oracle true set. Selection and the oracle use the SAME normLev metric, so it MUST -- if it does not,
# stop and print the offending IDs.
print('\nPartner-in-oracle invariant (24e):')
for feed in ['AA', 'SS', '3Di']:
    idx = ORACLE[feed]['idx']; ts = ORACLE[feed]['true_sets']
    hp = feed_high(feed)
    bad = []
    for a, b in zip(hp['domain_a'], hp['domain_b']):
        if a in ts and b in idx and idx[b] not in ts[a]: bad.append((a, b))
        if b in ts and a in idx and idx[a] not in ts[b]: bad.append((b, a))
    assert not bad, f'{feed}: {len(bad)} high-pair partner(s) missing from query oracle set: {bad[:10]}'
    print(f'  [{feed:<3}] all {len(hp)} high pairs: designated partner present in query oracle set  OK')


## 11. Retrieval metrics — per-query + bootstrap 95% CI (seed-fixed)

In [ ]:
K_LIST = (10, 50)
K_MAP  = 10
N_BOOT = 2000

def _ap_at_k(order, true_set, k):
    nt = len(true_set)
    if nt == 0: return np.nan
    hits = 0; ap = 0.0
    for i, oi in enumerate(order[:k], start=1):
        if oi in true_set:
            hits += 1; ap += hits / i
    return ap / min(nt, k)

def per_query_metrics(sim_fn, cache, k_list=K_LIST, k_map=K_MAP):
    idx = cache['idx']
    keys = ['ap', 'rprec', 'nt'] + [f'{m}@{k}' for k in k_list for m in ('rec', 'prec', 'hit')]
    per = {kk: [] for kk in keys}
    for q in cache['q_ids']:
        qi = idx[q]; ts = cache['true_sets'][q]; nt = len(ts)
        if nt == 0: continue
        sim = np.asarray(sim_fn(qi), dtype=float).copy(); sim[qi] = -np.inf
        order = np.argsort(-sim)
        per['nt'].append(nt)
        per['ap'].append(_ap_at_k(order, ts, k_map))
        per['rprec'].append(len(set(order[:nt].tolist()) & ts) / nt)
        for k in k_list:
            ret = len(set(order[:k].tolist()) & ts)
            per[f'rec@{k}'].append(ret / nt)
            per[f'prec@{k}'].append(ret / k)
            per[f'hit@{k}'].append(1.0 if ret >= 1 else 0.0)
    return {kk: np.asarray(v, float) for kk, v in per.items()}

def summarize_ci(per, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(per['nt'])
    out = {'n_q': n, 'median_n_true': float(np.median(per['nt'])) if n else np.nan}
    fields = {f'MAP@{K_MAP}': 'ap', 'Rprec': 'rprec', 'precision@10': 'prec@10',
              'recall@10': 'rec@10', 'recall@50': 'rec@50', 'hitrate@10': 'hit@10'}
    boot_idx = rng.integers(0, n, size=(n_boot, n)) if n >= 2 else None
    for name, kk in fields.items():
        arr = per[kk]; out[name] = float(arr.mean()) if n else np.nan
        if boot_idx is not None:
            b = arr[boot_idx].mean(axis=1)
            out[name + '_lo'] = float(np.percentile(b, 2.5))
            out[name + '_hi'] = float(np.percentile(b, 97.5))
        else:
            out[name + '_lo'] = out[name + '_hi'] = np.nan
    return out

def encoder_pool_np(model, feed):
    pe, _ = encode_pool(model, LOOKUP[feed], ORACLE[feed]['pool_ids'])
    return pe.cpu().numpy()

def length_pool(feed):
    return np.array([len(LOOKUP[feed][d]) for d in ORACLE[feed]['pool_ids']], dtype=float)

## 12. Compute — AUROC + encoder retrieval (with CI) + length-only baseline

In [ ]:
ENCODERS = [('AA-enc', model_aa), ('SS-enc', model_ss)]
FEEDS = ['AA', 'SS', '3Di']
results = []

for enc_label, model in ENCODERS:
    for feed in FEEDS:
        auroc, n_pos, n_tot = auroc_cell(model, feed)
        emb = encoder_pool_np(model, feed)
        per = per_query_metrics(lambda qi, e=emb: 1.0 - np.linalg.norm(e - e[qi], axis=1) / 2.0, ORACLE[feed])
        s = summarize_ci(per)
        row = {'encoder': enc_label, 'feed': feed,
               'role': 'in-domain' if (enc_label, feed) in [('AA-enc', 'AA'), ('SS-enc', 'SS')] else 'cross-rep',
               'auroc': auroc, 'auroc_n_pos': n_pos, 'auroc_n_total': n_tot}
        row.update(s); results.append(row)
        print(f"{enc_label} x {feed:<4} | AUROC={auroc:.3f} (n_pos={n_pos}) | "
              f"MAP@10={s['MAP@10']:.3f} [{s['MAP@10_lo']:.3f},{s['MAP@10_hi']:.3f}] | "
              f"hit@10={s['hitrate@10']:.3f} | med|T|={s['median_n_true']:.0f} (n_q={s['n_q']})")

print('\n--- length-only baseline ---')
for feed in FEEDS:
    plen = length_pool(feed)
    per = per_query_metrics(lambda qi, p=plen: 1.0 - np.abs(p - p[qi]) / np.maximum(p, p[qi]), ORACLE[feed])
    s = summarize_ci(per)
    row = {'encoder': 'length-only', 'feed': feed, 'role': 'baseline',
           'auroc': np.nan, 'auroc_n_pos': 0, 'auroc_n_total': 0}
    row.update(s); results.append(row)
    print(f"length   x {feed:<4} | MAP@10={s['MAP@10']:.3f} [{s['MAP@10_lo']:.3f},{s['MAP@10_hi']:.3f}] | "
          f"hit@10={s['hitrate@10']:.3f} | med|T|={s['median_n_true']:.0f}")

res_df = pd.DataFrame(results)

## 13. Summary table + CSV (feed-matched results of record)

In [ ]:
print('=' * 128)
print('COLAB24d SUMMARY — feed-matched retrieval (bootstrap 95% CI) + length-only baseline; exact-oracle relevance')
print('=' * 128)
print(f'{"enc":<12}{"feed":<6}{"role":<11}{"AUROC":>7}{"n_pos":>6}{"MAP@10 [95% CI]":>24}'
      f'{"hit@10":>9}{"rec@50":>8}{"med|T|":>8}{"n_q":>6}')
print('-' * 128)
for r in results:
    a = f'{r["auroc"]:.3f}' if not np.isnan(r["auroc"]) else '-'
    mp = f'{r["MAP@10"]:.3f} [{r["MAP@10_lo"]:.3f},{r["MAP@10_hi"]:.3f}]'
    print(f'{r["encoder"]:<12}{r["feed"]:<6}{r["role"]:<11}{a:>7}{r["auroc_n_pos"]:>6}{mp:>24}'
          f'{r["hitrate@10"]:>9.3f}{r["recall@50"]:>8.3f}{r["median_n_true"]:>8.0f}{r["n_q"]:>6}')
res_df.to_csv('colab24e_summary.csv', index=False)
print('\nSaved: colab24e_summary.csv  +  three eval tables in sampledata/cath/ '
      '(cath_eval_{aa,ss,3di}_perrep_24e.csv.gz)')

## 11. Chart 1 — Accuracy (AUROC is-high)

Unchanged from colab22. Grouped bars: blue = AA-encoder, green = SS-encoder, dashed = chance.
**AA bars are n_pos=5** (only 5 high-AA pairs exist) -> read AA as underpowered; SS (n=333) and
3Di (n=38) are the trustworthy columns.

In [ ]:
import matplotlib.pyplot as plt
COL_AA_ENC = '#1f77b4'   # blue  = AA-encoder
COL_SS_ENC = '#2ca02c'   # green = SS-encoder

def grouped_vals(metric):
    return ([next(r[metric] for r in results if r['encoder']=='AA-enc' and r['feed']==f) for f in FEEDS],
            [next(r[metric] for r in results if r['encoder']=='SS-enc' and r['feed']==f) for f in FEEDS])

aa_auroc, ss_auroc = grouped_vals('auroc')
x = np.arange(len(FEEDS)); w = 0.38
fig, ax = plt.subplots(figsize=(8.5, 5.2))
b1 = ax.bar(x - w/2, aa_auroc, w, label='AA-encoder', color=COL_AA_ENC)
b2 = ax.bar(x + w/2, ss_auroc, w, label='SS-encoder', color=COL_SS_ENC)
ax.axhline(0.5, ls='--', color='grey', lw=1)
ax.text(len(FEEDS)-0.5, 0.515, 'chance (0.5)', color='grey', fontsize=9, ha='right')
ax.bar_label(b1, fmt='%.3f', padding=2, fontsize=9)
ax.bar_label(b2, fmt='%.3f', padding=2, fontsize=9)
ax.set_ylim(0, 1.05); ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_ylabel('AUROC  (is-high >= 0.70)'); ax.set_xlabel('Feed modality')
ax.set_xticks(x); ax.set_xticklabels(FEEDS)
ax.set_title('Accuracy: can embedding distance separate a high-similarity pair from a random one?')
for i, f in enumerate(FEEDS):
    npos = next(r['auroc_n_pos'] for r in results if r['feed']==f)
    ax.annotate(f'n_pos={npos}', (i, 0), (0, -28), textcoords='offset points',
                ha='center', va='top', fontsize=8, color='dimgray')
ax.legend(loc='lower left', framealpha=0.9)
plt.tight_layout(); plt.savefig('colab24e_accuracy_auroc.png', dpi=150, bbox_inches='tight'); plt.show()

## 12. Chart 2 — Retrieval quality (MAP@10) vs the length-only baseline, bootstrap 95% CI

Three bars per feed: AA-encoder (blue), SS-encoder (green), **length-only baseline (grey)**, with
bootstrap 95% CIs. Where an encoder bar clears the grey one, the embedding is contributing
sequence-order signal beyond length-banding; where they overlap, the result is length-explained.

In [ ]:
def val_ci(metric, enc):
    pts = np.array([next(r[metric] for r in results if r['encoder'] == enc and r['feed'] == f) for f in FEEDS])
    lo  = np.array([next(r[metric + '_lo'] for r in results if r['encoder'] == enc and r['feed'] == f) for f in FEEDS])
    hi  = np.array([next(r[metric + '_hi'] for r in results if r['encoder'] == enc and r['feed'] == f) for f in FEEDS])
    return pts, lo, hi

x = np.arange(len(FEEDS)); w = 0.27
fig, ax = plt.subplots(figsize=(9.5, 5.5))
for off, enc, col, lab in [(-w, 'AA-enc', COL_AA_ENC, 'AA-encoder'),
                           (0.0, 'SS-enc', COL_SS_ENC, 'SS-encoder'),
                           (w, 'length-only', '#888888', 'length-only baseline')]:
    p, lo, hi = val_ci('MAP@10', enc)
    ax.bar(x + off, p, w, label=lab, color=col)
    ax.errorbar(x + off, p, yerr=np.vstack([p - lo, hi - p]), fmt='none', ecolor='black', capsize=3, lw=1)
ax.set_ylim(0, 1.05); ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_ylabel(f'MAP@{K_MAP}  (vs exact-Levenshtein neighbour set)')
ax.set_xlabel('Feed modality'); ax.set_xticks(x); ax.set_xticklabels(FEEDS)
ax.set_title('Retrieval quality vs a length-only baseline (bootstrap 95% CI)')
for i, f in enumerate(FEEDS):
    mt = next(r['median_n_true'] for r in results if r['feed'] == f)
    ax.annotate(f'med|T|={mt:.0f}', (i, 0), (0, -28), textcoords='offset points',
                ha='center', va='top', fontsize=8, color='dimgray')
ax.legend(loc='upper right', framealpha=0.9, fontsize=9)
plt.tight_layout(); plt.savefig('colab24e_retrieval_vs_length.png', dpi=150, bbox_inches='tight'); plt.show()

## 14. Embedding-space analysis — linear probes + PCA (primary), UMAP (illustration)

Per the review, the **original-space linear evidence is primary**; UMAP is demoted to illustration
because its axis claims are nonlinear and orientation-arbitrary.

- **14.2 Linear probes** — held-out Ridge: is length (and composition) *linearly recoverable* from
  the 128-d embedding? Pairs with the Section-10 length-only baseline — the probe says length is
  *encoded*; the baseline says whether that encoding *explains retrieval*.
- **14.3 PCA spectrum + axis correlations** — intrinsic dimensionality (isotropy) + which linear axis
  carries length/composition.
- **14.4 PCA-truncation retrieval** — rate-distortion: MAP@10 vs # PCs kept.
- **14.5–14.8 UMAP, neighbour tables, displacement probe, zoom** — illustration; the neighbour
  tables are original-space and are the hard qualitative evidence.

All on the **AA encoder / AA feed** (the well-posed case).

In [ ]:
import umap
from sklearn.decomposition import PCA
from scipy.stats import spearmanr

# ---- composition helpers (alphabet-agnostic content descriptors) ----
def comp_hist(seq):
    v = np.zeros(len(AA_ALPHABET))
    for c in seq:
        j = CHAR_TO_IDX.get(c, -1)
        if j >= 0: v[j] += 1
    s = v.sum()
    return v / s if s > 0 else v

def shannon_entropy(seq):
    h = comp_hist(seq); h = h[h > 0]
    return float(-(h * np.log2(h)).sum()) if h.size else 0.0

# ---- substrate: AA-encoder pool embedding + 2D UMAP + descriptors ----
VIZ_MODEL = model_aa
pool_ids_viz = POOL_IDS['AA']
idx_viz = {d: i for i, d in enumerate(pool_ids_viz)}
pool_emb_viz, _ = encode_pool(VIZ_MODEL, id_to_aa, pool_ids_viz)
pool_emb_viz = pool_emb_viz.cpu().numpy()
print(f'pool embeddings: {pool_emb_viz.shape}')

reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED)
pool_2d = reducer.fit_transform(pool_emb_viz)
id_to_xy = {d: pool_2d[i] for i, d in enumerate(pool_ids_viz)}

lens_viz = np.array([len(id_to_aa[d]) for d in pool_ids_viz])
comp_mat = np.stack([comp_hist(id_to_aa[d]) for d in pool_ids_viz])          # (N, 20)
ent_viz  = np.array([shannon_entropy(id_to_aa[d]) for d in pool_ids_viz])
comp_pc  = PCA(n_components=2, random_state=SEED).fit_transform(comp_mat)     # composition axes

HIGH_PAIRS_AA = [tuple(p) for p in feed_high('AA')[['domain_a', 'domain_b']].values]
print(f'high-AA pairs for overlay: {len(HIGH_PAIRS_AA)}')

### 14.2 Linear probes — what is *linearly recoverable* from the 128-d embedding?

Held-out 5-fold Ridge. A high length-R² makes "length is linearly recoverable" a sharp claim; read
it together with the Section-10 length-only retrieval baseline (encoded ≠ load-bearing).

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error

def linear_probe(emb, target, n_splits=5, seed=SEED):
    kf = KFold(n_splits, shuffle=True, random_state=seed)
    pred = np.zeros(len(target), dtype=float)
    for tr, te in kf.split(emb):
        pred[te] = Ridge(alpha=1.0).fit(emb[tr], target[tr]).predict(emb[te])
    return float(r2_score(target, pred)), float(mean_absolute_error(target, pred))

r2_len, mae_len = linear_probe(pool_emb_viz, lens_viz.astype(float))
r2_pc1, mae_pc1 = linear_probe(pool_emb_viz, comp_pc[:, 0])

kf = KFold(5, shuffle=True, random_state=SEED)
pred_h = np.zeros_like(comp_mat)
for tr, te in kf.split(pool_emb_viz):
    pred_h[te] = Ridge(alpha=1.0).fit(pool_emb_viz[tr], comp_mat[tr]).predict(pool_emb_viz[te])
r2_hist = float(r2_score(comp_mat, pred_h, multioutput='variance_weighted'))

print('Held-out linear probe (Ridge, 5-fold) — linear recoverability from the 128-d AA-encoder embedding')
print(f'  length                       : R2 = {r2_len:6.3f}   MAE = {mae_len:5.1f} residues')
print(f'  composition PC1              : R2 = {r2_pc1:6.3f}   MAE = {mae_pc1:5.3f}')
print(f'  full AA-composition histogram: R2 = {r2_hist:6.3f}   (variance-weighted, 20-d)')
print('\n-> length is (near-)linearly recoverable; composition only partially.')
print('   Section 10 length-only baseline tests whether that recoverability EXPLAINS retrieval')
print('   or is encoded-but-not-load-bearing.')

### 14.3 PCA — variance spectrum (intrinsic dimensionality) + linear axis correlations

Is the representation low-rank or distributed/isotropic? And which *linear* axis carries length vs
composition (the harder version of the colab23 UMAP-axis table).

In [ ]:
pca = PCA(n_components=min(128, pool_emb_viz.shape[1]), random_state=SEED).fit(pool_emb_viz)
evr = pca.explained_variance_ratio_; cum = np.cumsum(evr)
d90 = int(np.searchsorted(cum, 0.90) + 1); d99 = int(np.searchsorted(cum, 0.99) + 1)
pcs = pca.transform(pool_emb_viz)

print('PCA variance spectrum (AA-encoder pool embedding)')
print(f'  PC1 {evr[0]:.3f} | PC2 {evr[1]:.3f} | PC1+2 {cum[1]:.3f}')
print(f'  components for 90% var: {d90} ; for 99%: {d99}  (of {len(evr)})')
print(f'  -> {"low-rank" if d90 <= 8 else "distributed / isotropic"} representation')

def _rho(a, b): return float(spearmanr(a, b).correlation)
print('\nLinear (PCA) axis correlations:')
print(f'{"":<14}{"PC1":>8}{"PC2":>8}{"PC3":>8}')
for name, vec in [('length', lens_viz), ('comp-PC1', comp_pc[:, 0]), ('entropy', ent_viz)]:
    print(f'{name:<14}{_rho(pcs[:, 0], vec):>8.3f}{_rho(pcs[:, 1], vec):>8.3f}{_rho(pcs[:, 2], vec):>8.3f}')

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(np.arange(1, len(cum) + 1), cum, marker='o', ms=3)
ax.axhline(0.9, ls='--', color='grey', lw=1); ax.axvline(d90, ls=':', color='grey', lw=1)
ax.set_xscale('log', base=2)
ax.set_xlabel('# principal components'); ax.set_ylabel('cumulative explained variance')
ax.set_title('PCA variance spectrum (AA-encoder pool embedding)')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('colab24e_pca_spectrum.png', dpi=150, bbox_inches='tight'); plt.show()

### 14.4 PCA-truncation retrieval — rate-distortion (MAP@10 vs embedding rank)

Project pool embeddings onto the top-`d` PCs and recompute MAP@10. Saturation at small `d` = the
edit-distance retrieval signal is low-rank; a slow climb = distributed. AA feed (n_q=10) is
illustrative; SS feed is powered.

In [ ]:
D_GRID = [1, 2, 4, 8, 16, 32, 64, 128]

def trunc_map(model, feed, d_grid=D_GRID):
    emb = encoder_pool_np(model, feed)
    dmax = min(max(d_grid), emb.shape[1])
    full = PCA(n_components=dmax, random_state=SEED).fit_transform(emb)
    out = []
    for d in d_grid:
        sub = full[:, :d]
        per = per_query_metrics(lambda qi, s=sub: -np.linalg.norm(s - s[qi], axis=1), ORACLE[feed])
        out.append(float(per['ap'].mean()) if len(per['ap']) else np.nan)
    return out

map_aa = trunc_map(model_aa, 'AA')
map_ss = trunc_map(model_aa, 'SS')

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(D_GRID, map_aa, marker='o', color=COL_AA_ENC, label='AA feed (n_q=10, illustrative)')
ax.plot(D_GRID, map_ss, marker='s', color='#9467bd', label='SS feed (powered)')
ax.set_xscale('log', base=2); ax.set_xlabel('# PCA components kept'); ax.set_ylabel(f'MAP@{K_MAP}')
ax.set_title('Rate-distortion: retrieval quality vs embedding rank (AA-encoder)')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig('colab24e_pca_truncation.png', dpi=150, bbox_inches='tight'); plt.show()
print('truncation MAP@10  AA:', [f'{v:.2f}' for v in map_aa])
print('truncation MAP@10  SS:', [f'{v:.2f}' for v in map_ss])

### 14.5 UMAP figure — illustration only (high-AA pairs as red segments)

Kept for intuition; the axis claims here are the *soft* evidence — the linear probes (14.2-14.3) are
the hard version.

In [ ]:
ux, uy = pool_2d[:, 0], pool_2d[:, 1]
fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(ux, uy, c=lens_viz, cmap='viridis', s=8, alpha=0.5, linewidths=0)
plt.colorbar(sc, ax=ax, label='Protein length (AA)')
for a, b in HIGH_PAIRS_AA:
    if a in id_to_xy and b in id_to_xy:
        xa, ya = id_to_xy[a]; xb, yb = id_to_xy[b]
        ax.plot([xa, xb], [ya, yb], color='red', lw=1.5, alpha=0.9, zorder=5)
        ax.scatter([xa, xb], [ya, yb], color='red', s=80, edgecolors='black', linewidths=1.2, zorder=6)
        ax.annotate(f'{a[:6]}\u2194{b[:6]}', ((xa+xb)/2, (ya+yb)/2), fontsize=8, ha='center',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='red', alpha=0.8))
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
ax.set_title('AA-encoder pool embedding (UMAP) — high-AA partner pairs as red segments')
plt.tight_layout(); plt.savefig('colab24e_umap_pool.png', dpi=150, bbox_inches='tight'); plt.show()

### 14.6 Neighbourhood tables — original-space, the hard qualitative evidence

For each high-AA query, the top embedding neighbours with exact normLev + composition-sim. Shows the
true partner ranks #1 amid length-twins that are NOT exact-Levenshtein-similar.

In [ ]:
N_NB = 12
def neighbours_table(query, partner):
    qi = idx_viz[query]
    esim = 1.0 - np.linalg.norm(pool_emb_viz - pool_emb_viz[qi], axis=1) / 2.0
    esim[qi] = -np.inf
    order = np.argsort(-esim)[:N_NB]
    qpart_nl = norm_lev(id_to_aa[query], id_to_aa[partner])
    print(f'\nQuery {query} (len {len(id_to_aa[query])}) | partner {partner} '
          f'(exact normLev {qpart_nl:.3f})')
    print(f'  {"rank":<5}{"domain":<10}{"emb-sim":>8}{"len":>5}{"normLev":>9}{"comp-sim":>9}  note')
    for rk, oi in enumerate(order, 1):
        d = pool_ids_viz[oi]
        nl = norm_lev(id_to_aa[query], id_to_aa[d])
        cs = float((comp_mat[qi] * comp_mat[oi]).sum() /
                   max(1e-9, np.linalg.norm(comp_mat[qi]) * np.linalg.norm(comp_mat[oi])))
        note = '<== TRUE PARTNER' if d == partner else ('high-sim' if nl >= BAND_HIGH else '')
        print(f'  {rk:<5}{d:<10}{esim[oi]:>8.3f}{len(id_to_aa[d]):>5}{nl:>9.3f}{cs:>9.3f}  {note}')

for a, b in HIGH_PAIRS_AA[:3]:
    neighbours_table(a, b)

### 14.7 Horizontal-vs-vertical displacement probe — UMAP illustration

What changes when you move along each UMAP axis (length vs same-length content). Illustration; read
alongside the linear axis correlations in 14.3.

In [ ]:
rng_viz = np.random.default_rng(SEED)
N_PAIRS = 30000
ii = rng_viz.integers(0, len(pool_ids_viz), N_PAIRS)
jj = rng_viz.integers(0, len(pool_ids_viz), N_PAIRS)
ok = ii != jj; ii, jj = ii[ok], jj[ok]

dx   = np.abs(ux[ii] - ux[jj])
dy   = np.abs(uy[ii] - uy[jj])
dlen = np.abs(lens_viz[ii] - lens_viz[jj]).astype(float)

# composition distance, vectorised: 1 - cosine on the precomputed histograms
ca, cb = comp_mat[ii], comp_mat[jj]
na = np.linalg.norm(ca, axis=1); nb = np.linalg.norm(cb, axis=1)
denom = np.clip(na * nb, 1e-9, None)
dcomp = 1.0 - (ca * cb).sum(1) / denom

ratio = (dx + 1e-9) / (dy + 1e-9)
horiz = ratio > np.quantile(ratio, 0.8)   # top 20% most-horizontal displacements
vert  = ratio < np.quantile(ratio, 0.2)   # top 20% most-vertical

print('Displacement-direction probe (what changes when you move along each axis)')
print(f'{"group":<22}{"mean d-length":>14}{"mean d-comp":>13}')
print(f'{"mostly-horizontal":<22}{dlen[horiz].mean():>14.1f}{dcomp[horiz].mean():>13.3f}')
print(f'{"mostly-vertical":<22}{dlen[vert].mean():>14.1f}{dcomp[vert].mean():>13.3f}')
print(f'{"all pairs":<22}{dlen.mean():>14.1f}{dcomp.mean():>13.3f}')

same_len = dlen <= 3
print(f'\nAmong near-same-length pairs (|d-length| <= 3, n={int(same_len.sum())}):')
print(f'  corr(|dx|, d-comp) = {np.corrcoef(dx[same_len], dcomp[same_len])[0, 1]:+.3f}')
print(f'  corr(|dy|, d-comp) = {np.corrcoef(dy[same_len], dcomp[same_len])[0, 1]:+.3f}')
print('  -> at fixed length, the axis whose displacement tracks d-comp is the content axis;')
print('     i.e. two same-length sequences with totally different characters separate along THAT axis.')

### 14.8 Local zoom — illustration

One pair's neighbourhood coloured by exact normLev-to-query: the crowd of length-twins made visible.

In [ ]:
qa, qb = HIGH_PAIRS_AA[0]
cx, cy = id_to_xy[qa]
span_x = (ux.max() - ux.min()) * 0.12
span_y = (uy.max() - uy.min()) * 0.12
near = (np.abs(ux - cx) < span_x) & (np.abs(uy - cy) < span_y)
near_idx = np.where(near)[0]
nl_to_q = np.array([norm_lev(id_to_aa[qa], id_to_aa[pool_ids_viz[i]]) for i in near_idx])

fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(ux[near_idx], uy[near_idx], c=nl_to_q, cmap='magma', vmin=0, vmax=1,
                s=30, alpha=0.85, linewidths=0)
plt.colorbar(sc, ax=ax, label=f'exact normLev to query {qa}')
xa, ya = id_to_xy[qa]; xb, yb = id_to_xy[qb]
ax.plot([xa, xb], [ya, yb], color='red', lw=2, zorder=5)
ax.scatter([xa, xb], [ya, yb], color='red', s=130, edgecolors='black', linewidths=1.3, zorder=6)
ax.annotate('query', (xa, ya), fontsize=9, color='red')
ax.annotate('partner', (xb, yb), fontsize=9, color='red')
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
ax.set_title(f'Local neighbourhood of {qa} <-> {qb}\n(most nearby points are length-twins with low normLev)')
plt.tight_layout(); plt.savefig('colab24e_umap_zoom.png', dpi=150, bbox_inches='tight'); plt.show()
print(f'local window: {len(near_idx)} points; '
      f'{int((nl_to_q >= BAND_HIGH).sum())} of them are genuine high-sim (>= {BAND_HIGH})')

## How to read this notebook

- **§6 + §6b are the 24e deliverable:** three feed-matched eval tables, each scored by the *same*
  `current_normlev` the oracle uses, now carrying **both** supplied source scores for provenance, plus
  an audit that quantifies the supplied↔recomputed drift and the 0.70 flips. The legacy
  `cath_eval.csv.gz` and the 24d `*_perrep` tables are untouched (new files use the `_24e` suffix).
- **§7 audit / §10 oracle asserts** are the receipts: every feed-matched query has ≥1 oracle
  neighbour, and every high pair's designated partner is itself in that query's oracle set (same
  metric on both sides).
- **§11–§14 (restored from colab24c):** the decisive retrieval read is per feed — if the encoder
  MAP@10 bar clears the grey length-only bar (non-overlapping CIs), the embedding contributes
  **sequence-order** signal beyond length-banding; if they overlap, that feed is **length-explained**.
  In §14 the original-space linear probes + PCA are **primary** (held-out length R² = how much length
  is *linearly encoded*; the length-only baseline = whether that encoding *drives retrieval*); UMAP,
  the displacement probe, and the zoom are **illustration**; the neighbour tables are the hard
  qualitative evidence. All §14 panels use the **AA encoder / AA feed** (the well-posed case), with the
  high-AA overlay taken from the feed-matched AA eval table.
- **Comparability discipline:** the model, pools, query logic, oracle, and metrics are byte-identical
  to 24d. Read any difference from 24d only as the effect of the added provenance/audit/figures — never
  as the model improving or regressing. Relevance is the exact-Levenshtein high string-similarity
  neighbour set; **no biological claim is made**.
